# Convolutional Neural Networks (CNN)

Phân loại ảnh biểu tượng giao thông 28×28 (6 lớp) bằng CNN trên Keras.

**Quy trình**
1. Load dữ liệu
2. Chuẩn hóa, reshape, one-hot encoding
3. Tách train / validation
4. Xây dựng mô hình CNN
5. Compile, tăng cường dữ liệu và huấn luyện
6. Đánh giá

In [2]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPool2D
from keras.models import Sequential
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings("ignore")

DATA_DIR = Path("datasets/transportation_cnn")
if not DATA_DIR.exists():
    DATA_DIR = Path("../datasets/transportation_cnn")
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_DIR.resolve()}. Chạy notebook từ week2 hoặc PRN501-CNN-report."
    )

print("Dataset files:", sorted(os.listdir(DATA_DIR)))

FileNotFoundError: Không tìm thấy C:\Users\admin\datasets\transportation_cnn. Chạy notebook từ week2 hoặc PRN501-CNN-report.

## 1. Load dữ liệu

Mỗi dòng là một ảnh xám 28×28 (784 pixel) kèm nhãn `label` (0–5).

In [ ]:
train = pd.read_csv(DATA_DIR / "transportation_train.csv")
test = pd.read_csv(DATA_DIR / "transportation_test.csv")

Y_train = train["label"]
X_train = train.drop(columns=["label"])
Y_test = test["label"]
X_test = test.drop(columns=["label"])

print("train:", train.shape, "| test:", test.shape)
print(Y_train.value_counts().sort_index())
train.head()

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(x=Y_train, palette="icefire")
plt.title("Số mẫu theo lớp (train)")
plt.show()

img = X_train.iloc[0].to_numpy().reshape(28, 28)
plt.imshow(img, cmap="gray")
plt.title(f"Mẫu label = {Y_train.iloc[0]}")
plt.axis("off")
plt.show()

## 2. Chuẩn hóa, reshape, one-hot encoding

- Chia pixel cho 255 để CNN hội tụ nhanh hơn.
- Đưa ảnh về `(28, 28, 1)` — Keras cần chiều kênh (ảnh xám = 1 kênh).
- One-hot nhãn: ví dụ `2 → [0, 0, 1, 0, 0, 0]`.

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.values.reshape(-1, 28, 28, 1)
X_test = X_test.values.reshape(-1, 28, 28, 1)

Y_train = to_categorical(Y_train, num_classes=6)
Y_test = to_categorical(Y_test, num_classes=6)

print("X_train:", X_train.shape, "| Y_train:", Y_train.shape)
print("X_test :", X_test.shape, "| Y_test :", Y_test.shape)

## 3. Tách train / validation

Giữ 10% tập train làm validation khi huấn luyện.

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.1, random_state=2
)

print("X_train:", X_train.shape, "| X_val:", X_val.shape)
print("Y_train:", Y_train.shape, "| Y_val:", Y_val.shape)

## 4. Xây dựng mô hình CNN

Luồng CNN: **Conv2D → MaxPool → Dropout → Conv2D → MaxPool → Dropout → Flatten → Dense → Softmax**.

- **Conv2D**: lọc đặc trưng (cạnh, hình khối).
- **Same padding**: giữ kích thước feature map.
- **MaxPool**: giảm kích thước, giảm overfitting.
- **Flatten + Dense**: đưa đặc trưng vào lớp fully connected để phân loại 6 lớp.

In [ ]:
model = Sequential([
    Conv2D(filters=8, kernel_size=(5, 5), padding="Same", activation="relu", input_shape=(28, 28, 1)),
    MaxPool2D(pool_size=(2, 2)),
    Dropout(0.25),
    Conv2D(filters=16, kernel_size=(3, 3), padding="Same", activation="relu"),
    MaxPool2D(pool_size=(2, 2), strides=(2, 2)),
    Dropout(0.25),
    Flatten(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(6, activation="softmax"),
])

model.summary()

## 5. Compile, data augmentation và huấn luyện

- Optimizer **Adam**, loss **categorical_crossentropy** (đa lớp).
- Data augmentation xoay / zoom / dịch nhẹ để giảm overfitting.

In [ ]:
optimizer = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])

epochs = 10
batch_size = 250

datagen = ImageDataGenerator(
    rotation_range=5,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=False,
    vertical_flip=False,
)
datagen.fit(X_train)

history = model.fit(
    datagen.flow(X_train, Y_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=(X_val, Y_val),
    steps_per_epoch=X_train.shape[0] // batch_size,
)

## 6. Đánh giá

Theo dõi loss trên validation và ma trận nhầm lẫn.

In [ ]:
# ==============================================================================
# NGƯỜI 4: DỰ ĐOÁN VÀ ĐÁNH GIÁ MÔ HÌNH (PREDICTION & EVALUATION)
# ==============================================================================

# 1. Trực quan hóa Validation Loss qua từng Epoch
# Giúp theo dõi quá trình học của mô hình và phát hiện dấu hiệu Overfitting (nếu val_loss bắt đầu tăng lại)
plt.plot(history.history["val_loss"], color="b", label="validation loss")
plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# 2. Thực hiện dự đoán trên tập dữ liệu Validation (X_val)
# Mô hình trả về một mảng chứa 6 giá trị xác suất (Probability) tương ứng với 6 lớp cho mỗi ảnh
Y_pred = model.predict(X_val)

# 3. Sử dụng hàm argmax để lấy chỉ số (index) của lớp có xác suất dự đoán lớn nhất
Y_pred_classes = np.argmax(Y_pred, axis=1)

# 4. Chuyển nhãn thực tế từ dạng One-hot Encoding về lại dạng chỉ số lớp (Integer Class)
Y_true = np.argmax(Y_val, axis=1)

# 5. Tạo ma trận nhầm lẫn (Confusion Matrix) để so sánh các cặp lớp thực tế (Y_true) và lớp dự đoán (Y_pred_classes)
confusion_mtx = confusion_matrix(Y_true, Y_pred_classes)

# 6. Trực quan hóa ma trận nhầm lẫn dưới dạng biểu đồ nhiệt (Heatmap)
plt.figure(figsize=(8, 8))
sns.heatmap(
    confusion_mtx,
    annot=True,         # Hiển thị số lượng mẫu chi tiết ngay trên từng ô của ma trận
    fmt="d",            # Định dạng kiểu hiển thị số nguyên
    cmap="Greens",      # Tông màu hiển thị xanh lá
    linewidths=0.01,    # Độ rộng đường ranh giới giữa các ô
    linecolor="gray",   # Màu đường ranh giới giữa các ô
)
plt.xlabel("Predicted") # Trục hoành (X-axis) biểu thị lớp mô hình dự đoán
plt.ylabel("True")      # Trục tung (Y-axis) biểu thị lớp thực tế (Ground Truth)
plt.title("Confusion Matrix")
plt.show()

NameError: name 'plt' is not defined